# Setup
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv

dotenv.load_dotenv()
AA_PG_CONNECTION_STRING = os.getenv("AA_PG_CONNECTION_STRING")
AA_GEMINI_API_KEY = os.getenv("AA_GEMINI_API_KEY")

In [ ]:
from langchain_postgres import PGEngine
from sqlalchemy.ext.asyncio import create_async_engine

# エンジン初期化
assert AA_PG_CONNECTION_STRING is not None
assert AA_GEMINI_API_KEY is not None

sa_engine = create_async_engine(AA_PG_CONNECTION_STRING, pool_size=5)
pg_engine = PGEngine.from_engine(sa_engine)

# Extract

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from assistant_agent.loaders import MarkdownLoader

# Vault から読み込み
vault_path = Path("../../docs/dataset_website").resolve()
loader = DirectoryLoader(
    str(vault_path),
    glob="**/*.md",
    loader_cls=MarkdownLoader,  # type: ignore[arg-type]
)
docs = loader.load()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
from sqlalchemy import text
from assistant_agent.entities.base import VaultUtils
from assistant_agent.entities.postgres import AssetBase, AppBase, SampleEntity

# DB へ取り込み
async with sa_engine.begin() as conn:
    await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
    await conn.run_sync(AssetBase.metadata.create_all)
    await conn.run_sync(AppBase.metadata.create_all)
await VaultUtils.sync_docs(docs[:10], sa_engine, SampleEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
import sqlalchemy
from sqlalchemy.ext.asyncio import AsyncSession
from assistant_agent.entities.postgres import SampleEntity

async with AsyncSession(sa_engine) as sess:
    res = await sess.execute(sqlalchemy.select(SampleEntity).limit(10))

list(res)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
res = splitter.split_documents(docs[:2])
print("\n=====================\n".join([item.page_content for item in res]))

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pydantic import SecretStr

from assistant_agent.entities.postgres import SampleEntity, SampleChunkEntity
from assistant_agent.services import VaultSampleRetriever
from assistant_agent.store import PostgresStoreConnector

assert AA_PG_CONNECTION_STRING is not None
assert AA_GEMINI_API_KEY is not None

store_conn = PostgresStoreConnector(AA_PG_CONNECTION_STRING)
sa_engine = store_conn.get_engine()
sample_retriever = VaultSampleRetriever(
    vault_entity=SampleEntity,
    chunk_entity=SampleChunkEntity,
    store_conn=store_conn,
    splitter=splitter,
    embed_model=GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        api_key=SecretStr(AA_GEMINI_API_KEY),
    ),
)

In [ ]:
await sample_retriever.sync_chunks()

# Retrieval

In [ ]:
chunks = await sample_retriever.search_documents("BPM", 5)
for item in chunks:
    print(item.page_content)
    print("==================")